# Register Foundry Model - API
## Step 1: Create the External Model Endpoint
Registers an Azure AI Foundry deployment as a Databricks external model endpoint using the OpenAI provider with standard Azure API key authentication (`openai_api_type: "azure"`).

**Configuration:**
- `openai_api_type`: `"azure"` — key-based Azure OpenAI auth
- `openai_api_key`: Stored in the Databricks-backed secret scope. 
- `openai_api_base`: The Azure AI Foundry resource endpoint URL.
- `openai_deployment_name`: Must match the deployment name in Foundry 
- `openai_api_version`: Must be a version supported by the Foundry resource 

**Notes:**
- Secrets are stored in a Databricks-backed scope (`foundry-creds`) rather than an Azure Key Vault-backed scope, as Model Serving cannot resolve KV-backed secrets.

In [0]:
import mlflow.deployments

client = mlflow.deployments.get_deploy_client("databricks")
workspace_url = "https://<workspace url>.azuredatabricks.net"  # Replace with your Databricks workspace URL. For example adb-1234567890123456.11
endpoint_name = "azure-gpt56-luna-api"  # You choose the name that will display in the Serving Endpoints listing in Databricks

dbutils.widgets.text("endpoint_name", endpoint_name)

client.create_endpoint(
    name=endpoint_name,
    config={
        "served_entities": [{
            "name": "gpt-56-luna-deploymment-name", # Deployment Name of the model in Foundry
            "external_model": {
                "name": "gpt-5.6-luna", # Must match the model name in Foundry
                "provider": "openai",
                "task": "llm/v1/chat",
                "openai_config": {
                    "openai_api_type": "azure",
                    "openai_api_key": "{{secrets/foundry-creds/foundry_api_key}}",
                    "openai_api_base": "https://<foundry project name>-resource.services.ai.azure.com",
                    "openai_deployment_name": "gpt-5.6-luna-deployment-name", # Deployment Name of the model in Foundry
                    "openai_api_version": "2024-12-01-preview",
                }
            }
        }]
    }
)

In [0]:
%skip
import mlflow.deployments
client = mlflow.deployments.get_deploy_client("databricks")
client.delete_endpoint(endpoint_name)
# Then re-run Cell 2 to recreate with fresh secret resolution

## Step 2: Verify the Endpoint is Ready
Calls `get_endpoint()` and prints the `state` field. The endpoint should report `{'ready': 'READY'}` within seconds of creation, confirming the configuration was accepted and the endpoint is available for queries.

In [0]:
import mlflow.deployments

endpoint = mlflow.deployments.get_deploy_client("databricks").get_endpoint(endpoint_name)
print(endpoint["state"])
# Expected: {"ready": "READY"}

## Step 3: Query the Endpoint
Uses `ai_query()` in SQL to send a prompt to the registered endpoint. The `:endpoint_name` parameter is bound to the widget value set in Step 1.

In [0]:
%sql
SELECT ai_query(
  :endpoint_name,
  request => "What is the population of Los Angeles?"
)